# 15 · Capstone — an End-to-End BrewBox Pipeline 🏁

Time to put the whole track together. In this one notebook you'll build a
complete, production-shaped **Medallion pipeline** on the BrewBox data:

**Ingest (Auto Loader) → Bronze → Silver (clean + validate) → Gold (star schema +
mart) → serve**, with **data-quality gates** and notes on scheduling it as a job.

Everything uses the shared dataset from notebook 6 and writes to `cap_*` tables so
it's self-contained and won't disturb your earlier work. This is exactly the flow
a data engineer builds on the job.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
from pyspark.sql import functions as F, Window
spark.sql("USE SCHEMA brewbox")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
LANDING = f"/Volumes/{CATALOG}/brewbox/landing"
print("catalog:", CATALOG, "| landing:", LANDING)

## Step 1 · Ingest → Bronze (Auto Loader)

Incrementally load the raw `orders` files into a Bronze table with ingestion
metadata. Re-running is safe (the checkpoint tracks processed files).

In [ ]:
ckpt = f"{LANDING}/_checkpoints/cap_orders_bronze"
schema_loc = f"{LANDING}/_schemas/cap_orders"

q = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .load(f"{LANDING}/orders")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream.format("delta")
    .option("checkpointLocation", ckpt)
    .trigger(availableNow=True)
    .toTable("brewbox.cap_orders_bronze"))
q.awaitTermination()
print("Bronze rows:", spark.table("brewbox.cap_orders_bronze").count())

## Step 2 · Bronze → Silver (clean, dedupe, enrich, validate)

Type and standardize the data, keep one row per order, enrich with customer and
store context, and **validate** before publishing.

In [ ]:
bronze = spark.table("brewbox.cap_orders_bronze")
w = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())

silver = (bronze
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("status", F.lower(F.trim("status")))
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
    .join(spark.table("brewbox.customers").select("customer_id","country","loyalty_tier"), "customer_id", "left")
    .join(spark.table("brewbox.stores").select("store_id","region"), "store_id", "left")
    .withColumn("is_completed", F.col("status") == "completed")
    .select("order_id","customer_id","store_id","country","loyalty_tier","region",
            "order_ts","order_date","status","is_completed","amount"))

# Data-quality GATE: stop the pipeline if core rules fail
bad_keys = silver.filter("order_id IS NULL").count()
bad_amt  = silver.filter("amount < 0").count()
assert bad_keys == 0, f"{bad_keys} null order_id rows"
assert bad_amt == 0, f"{bad_amt} negative amounts"
silver.write.format("delta").mode("overwrite").saveAsTable("brewbox.cap_orders_silver")
print("Silver rows:", spark.table("brewbox.cap_orders_silver").count(), "| DQ gate passed")

## Step 3 · Silver → Gold (star schema)

Build the dimensions and the fact table, then an aggregated mart. This is what BI
and ML consume.

In [ ]:
# Dimensions
spark.table("brewbox.customers").write.format("delta").mode("overwrite").saveAsTable("brewbox.cap_dim_customer")
spark.table("brewbox.stores").write.format("delta").mode("overwrite").saveAsTable("brewbox.cap_dim_store")
spark.table("brewbox.products").write.format("delta").mode("overwrite").saveAsTable("brewbox.cap_dim_product")

# Fact (order grain)
(spark.table("brewbox.cap_orders_silver")
    .select("order_id","customer_id","store_id","order_date","status","is_completed","amount")
    .write.format("delta").mode("overwrite").saveAsTable("brewbox.cap_fct_orders"))

# Gold mart: daily completed revenue by region
(spark.table("brewbox.cap_fct_orders").filter("is_completed")
    .join(spark.table("brewbox.cap_dim_store").select("store_id","region"), "store_id")
    .groupBy("order_date","region")
    .agg(F.count("*").alias("orders"), F.round(F.sum("amount"),2).alias("revenue"))
    .write.format("delta").mode("overwrite").saveAsTable("brewbox.cap_mart_daily_revenue"))

print("Gold built: cap_fct_orders, cap_dim_*, cap_mart_daily_revenue")

## Step 4 · Serve — answer business questions

The whole point: fast, clean answers from the Gold layer.

In [ ]:
print("Top regions by completed revenue:")
(spark.table("brewbox.cap_mart_daily_revenue")
    .groupBy("region").agg(F.round(F.sum("revenue"),2).alias("revenue"))
    .orderBy(F.desc("revenue")).show())

print("Monthly revenue trend:")
(spark.table("brewbox.cap_mart_daily_revenue")
    .withColumn("month", F.date_format("order_date","yyyy-MM"))
    .groupBy("month").agg(F.round(F.sum("revenue"),2).alias("revenue"))
    .orderBy("month").show())

## Step 5 · Productionize it

You've built the pipeline; here's how it becomes a real, running product:

- **Schedule** it (notebook 13): a Databricks **Job** with tasks
  `ingest → silver → gold`, a daily cron, retries, and failure alerts — or deploy
  it as code with an **Asset Bundle**.
- **Or go declarative** (notebook 12): rewrite the three steps as a **Lakeflow
  Declarative Pipeline** with `@dlt.table` + **expectations**, and let Databricks
  manage dependencies, incremental updates, and data-quality metrics.
- **Optimize** (notebook 14): `OPTIMIZE`/liquid-cluster the big tables, broadcast
  small dims, and run on serverless/job compute with auto-termination.
- **Incremental, not full rebuild:** swap the `overwrite` writes for `MERGE`
  (notebook 8) keyed on `order_id`, and use SCD2 for changing dimensions.

## 🎓 Congratulations — you've completed the Databricks DE track

Across notebooks 6–15 you learned the platform & Unity Catalog, ingestion with
Auto Loader, Delta Lake in depth, Bronze→Silver→Gold transformations & modeling,
Structured Streaming, declarative pipelines with data quality, orchestration, and
performance — and tied it all together here.

**Extension challenges (try these):**
1. Add `cap_fct_order_items` (line grain) joining `order_items` + `products`, and a
   category-revenue mart.
2. Make the Silver load **incremental** with `MERGE` instead of `overwrite`.
3. Turn `cap_dim_customer` into an **SCD Type 2** dimension (notebook 8).
4. Add three **data-quality expectations** and convert the pipeline to **Lakeflow
   Declarative Pipelines** (notebook 12).
5. Wrap it in a scheduled **Job** with a `run_date` parameter (notebook 13).

Happy engineering! 🚀